# HAM10000 期末 05：網頁版原型驗證

不需要 GPU，一般執行階段即可。

原型改為靜態網頁版，模型轉成 ONNX 格式，在使用者的瀏覽器中執行。這份確認網頁版用的 model.onnx 和 02 的模型結果一致。

**這份做兩件事**
1. **預測一致**：用 model.onnx 重新預測種子 42 的整份測試集，和 02 存下的預測比對。
2. **熱圖一致**：網頁版的熱圖由 model.onnx 直接輸出，抽 20 張影像，和 04 用 PyTorch 算的 Grad-CAM 比對。

網頁版的影像縮放程式移植自 PIL，Claude 已確認兩者逐像素相同，所以這裡用 PIL 做前處理即可代表網頁版。不會改動任何結果檔。

## 1. 設定

執行前，請把 Claude 給的 **model.onnx** 放到雲端硬碟的 ham10000_project/prototype_web 資料夾。

In [ ]:
import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/ham10000_project'
ONNX = os.path.join(BASE, 'prototype_web', 'model.onnx')
PT = os.path.join(BASE, 'results_final_cnn', 'models', 'model__cw-aug__lesion__seed42.pt')
PRED = os.path.join(BASE, 'results_final_cnn', 'predictions', 'prob__cw-aug__lesion__seed42.npz')
SPLIT = os.path.join(BASE, 'results_final_baseline', 'split_assignment.csv')
for p in [ONNX, PT, PRED, SPLIT]:
    assert os.path.exists(p), f'找不到：{p}'
print('檔案都在，model.onnx 大小：', round(os.path.getsize(ONNX) / 1e6, 1), 'MB')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnxruntime'], check=True)

import numpy as np, pandas as pd, time, torch, torch.nn as nn, onnxruntime as ort
from PIL import Image
from torchvision import models
CLS = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']; MEL = CLS.index('mel')
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32); STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def preprocess(img):
    """與 02_cnn、原型相同：轉 RGB → 雙線性縮放為 224 × 224 → 除以 255 → ImageNet 正規化"""
    arr = np.asarray(img.convert('RGB').resize((224, 224), Image.BILINEAR), dtype=np.uint8)
    x = (arr.astype(np.float32) / 255.0 - MEAN) / STD
    return x.transpose(2, 0, 1)[None]

sess = ort.InferenceSession(ONNX, providers=['CPUExecutionProvider'])
print('model.onnx 載入完成，輸出：', [o.name for o in sess.get_outputs()])

## 2. 下載 HAM10000 影像

和 04 一樣用 kagglehub 下載，第一次約需幾分鐘。

In [ ]:
import kagglehub
kag = kagglehub.dataset_download('kmader/skin-cancer-mnist-ham10000')
HAM = {f[:-4]: os.path.join(r, f) for r, _, fs in os.walk(kag) for f in fs if f.endswith('.jpg') and f.startswith('ISIC_')}
print('HAM10000 影像數：', len(HAM))
assert len(HAM) == 10015

## 3. 預測一致

判定標準和 04 相同：每張影像七類機率的最大差異在 0.02 以內，且預測類別一致的比例達 99% 以上。

In [ ]:
z = np.load(PRED)
ids = z['test_image_id'].astype(str); ref = z['test']
probs = np.zeros_like(ref)
t0 = time.time()
for i, iid in enumerate(ids):
    probs[i] = sess.run(['probs'], {'input': preprocess(Image.open(HAM[iid]))})[0][0]
print(f'{len(ids)} 張，耗時 {time.time() - t0:.0f} 秒')
diff = np.abs(probs - ref).max(1)
agree = (probs.argmax(1) == ref.argmax(1)).mean()
print(f'最大機率差異：{diff.max():.4f}（中位數 {np.median(diff):.5f}）')
print(f'預測類別一致比例：{agree:.4f}')
assert diff.max() < 0.02 and agree >= 0.99, '網頁版與 02 的預測不一致，請停下來告訴組長'
print('驗證通過：網頁版的 model.onnx 與 02 的模型預測一致')

rows = []
for iid in ['ISIC_0026086', 'ISIC_0026077', 'ISIC_0025316']:
    k = list(ids).index(iid)
    rows.append({'影像': iid, '網頁版 黑色素瘤機率': round(float(probs[k, MEL]), 3), '02 黑色素瘤機率': round(float(ref[k, MEL]), 3),
                 '網頁版 預測類別': CLS[probs[k].argmax()]})
print(pd.DataFrame(rows).to_string(index=False))

## 4. 熱圖一致

用 04 的 PyTorch Grad-CAM 和網頁版的熱圖比對，三張示範影像加上隨機 17 張測試影像（亂數種子 42），每張比對預測類別與黑色素瘤兩張熱圖。熱圖數值在 0 到 1 之間，差異在 0.02 以內即通過。

In [ ]:
m = models.efficientnet_b0(weights=None)
m.classifier[1] = nn.Linear(m.classifier[1].in_features, 7)
m.load_state_dict(torch.load(PT, map_location='cpu', weights_only=True)); m.eval()

def to_224(cam):
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return np.asarray(Image.fromarray(np.uint8(cam * 255)).resize((224, 224), Image.BILINEAR)) / 255.0

def torch_cam(x, c):
    """與 04、原型 app.py 的 grad_cam 相同"""
    acts, grads = {}, {}
    def hook(mod, i, o):
        acts['a'] = o; o.register_hook(lambda g: grads.__setitem__('g', g))
    h = m.features[-1].register_forward_hook(hook)
    try:
        m.zero_grad(); out = m(torch.from_numpy(x)); out[0, c].backward()
    finally:
        h.remove()
    w = grads['g'].mean(dim=(2, 3), keepdim=True)
    return to_224(torch.relu((w * acts['a']).sum(1)).squeeze().detach().numpy())

def web_cam(cams, c):
    """網頁版：直接取 model.onnx 輸出的該類 CAM，ReLU 後正規化"""
    return to_224(np.maximum(cams[0, c], 0))

rng = np.random.default_rng(42)
pick = ['ISIC_0026086', 'ISIC_0026077', 'ISIC_0025316'] + list(rng.choice([i for i in ids if i not in ('ISIC_0026086', 'ISIC_0026077', 'ISIC_0025316')], 17, replace=False))
worst = 0
for iid in pick:
    x = preprocess(Image.open(HAM[iid]))
    p, cams = sess.run(['probs', 'cams'], {'input': x})
    for c in {int(p[0].argmax()), MEL}:
        worst = max(worst, np.abs(torch_cam(x, c) - web_cam(cams, c)).max())
print(f'{len(pick)} 張影像，熱圖最大差異：{worst:.4f}（0 到 1 之間）')
assert worst < 0.02, '網頁版與 PyTorch 的熱圖不一致，請停下來告訴組長'
print('驗證通過：網頁版熱圖與 PyTorch Grad-CAM 一致')